In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import os
import numpy as np
import json
from collections import Counter
from numpy import linalg as la
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import OPTICS
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import importlib
import model as model_module
importlib.reload(model_module)
LogKG = model_module.LogKG
from d1_adapter import build_d1_cases

# Exp config
CONFIG_PATH = os.path.join("../data", "config.json")
CASE_DIR = "../data/CMCC_case"
D1_DIR = "../data/D1"


In [ ]:
# Set the KGE of templates
EMBEDDING_SIZE = 16
template_embedding = None

# Dataset switch: 'CMCC' or 'D1'
DATASET_NAME = "D1"

# D1 config
D1_LOG_FILE = "preliminary_sel_log_dataset.csv"
D1_LABEL_FILE = "preliminary_train_label_dataset_s.csv"
D1_HISTORY_HOURS = 24
D1_USE_SERVER_MODEL = True


In [ ]:
# Get case label
def get_case_truth_label(case_name_list, input_config):
    truth_label_list = np.zeros((len(case_name_list)), dtype=int)
    label_config = {}
    for index, fault_class in enumerate(input_config.keys()):
        label_config[fault_class] = index
        for case_name in input_config[fault_class]:
            truth_label_list[case_name_list.index(case_name)] = index
    return truth_label_list


In [ ]:
# min samples in one cluster
min_samples = 3

def build_cluster_model():
    return OPTICS(min_samples=min_samples, metric="cosine", xi=0.05, algorithm="brute")

# IDF threshold
IDF_threshold = 0.4

# classifier
clf = RandomForestClassifier
clf_kwargs = {"random_state": 42}


In [ ]:
# =========================
# 1) 计算 EDM: Euclidean Distance Matrix（欧氏距离矩阵）
# 输入:
#   X: shape=(n_samples, n_features)
# 输出:
#   D: shape=(n_samples, n_samples)
#       D[i, j] 表示样本 i 和样本 j 的“欧氏距离”
# 注意:
#   函数名叫 compute_squared_EDM_method，但这里并没有平方（不是 squared distance）
#   la.norm 默认是 L2 范数 => sqrt(sum((xi-xj)^2))
# =========================
def compute_squared_EDM_method(X):
    n, m = X.shape
    D = np.zeros([n, n])

    # 双重循环，计算上三角（i < j），再对称赋值到下三角
    for i in range(n):
        for j in range(i + 1, n):
            # 计算两点欧氏距离
            D[i, j] = la.norm(X[i, :] - X[j, :])
            D[j, i] = D[i, j]
    return D


# =========================
# 2) 找“类中心点”的索引（类似 medoid）
# 思路:
#   - 先算簇内所有点两两距离矩阵 D
#   - 对每个点，把它到其它点的距离求和（越小越“居中”）
#   - 返回距离和最小的那个点的下标
# 输入:
#   cluster_embedding: shape=(k, embedding_dim)，某个簇里的 k 个点
# 输出:
#   返回簇内的“中心点”在 cluster_embedding 里的行号（0~k-1）
# =========================
def get_centroid_index(cluster_embedding):
    # 对每一行求和: distance_array[i] = sum_j D[i, j]
    distance_array = np.sum(compute_squared_EDM_method(cluster_embedding), axis=1)
    # 返回距离和最小的点（簇内最代表性的样本）
    return np.argmin(distance_array)


# =========================
# 3) 训练集聚类 + 选每个簇的代表样本（centroid/medoid）
# 输入:
#   train_set: 当前 fold 的训练 embedding（shape=(n_train, emb_dim)）
#   train_index: 这些训练样本在“全数据”里的原始索引列表
# 输出:
#   centroids: 每个簇选出来的代表样本的“原始索引”（对齐全数据）
#   cluster_: 聚类结果，长度 n_train，每个元素是簇 id（0..class_num-1），可能会有 -1 噪声
# 注意:
#   - 代码里把 -1 当噪声，但 class_num = max(cluster_)+1 会忽略 -1（不计为类别）
#   - 若存在 -1，后续 list comprehension 只遍历 0..class_num-1，不会处理噪声点
# =========================
def get_logkg_result(train_set, train_index, verbose=False):
    cluster_model = build_cluster_model()
    # 聚类：返回每个样本所属簇编号
    cluster_ = cluster_model.fit_predict(train_set)

    # 类别数量（不包含 -1 噪声）
    class_num = np.max(cluster_) + 1

    if verbose:
        print(cluster_)
        print(Counter(cluster_))
        print("class_num: {}".format(class_num))

    # 对每个簇 i：
    #   1) 找出 train_set 中属于该簇的样本下标集合 idx_in_train
    #   2) 在该簇内部找“中心点”（簇内距离和最小）
    #   3) 把它映射回全数据索引 train_index[...]
    centroids = [
        train_index[
            np.where(cluster_ == i)[0][
                get_centroid_index(train_set[np.where(cluster_ == i)[0]])
            ]
        ]
        for i in range(class_num)
    ]
    return centroids, cluster_


# =========================
# 4) 单个 fold 的训练/测试流程
# 流程:
#   1) 用 LogKG 计算 train/test 的 case embedding（每个 case 一个向量）
#   2) 在 train embedding 上聚类
#   3) 每个簇选一个代表样本，用其真实标签当作该簇的标签（pseudo-label）
#   4) 用这些 pseudo-label 样本训练一个分类器（clf）
#   5) 用分类器预测 test，返回 acc / macro_f1
# =========================
def LogKG_exp_run(case_name_list, case_truth_label,
                  train_index, test_index,
                  logkg_config, verbose=False):
    logkg = logkg_config

    # split 不能为空
    if len(train_index) == 0 or len(test_index) == 0:
        raise ValueError("train_index/test_index is empty. Please set a non-empty split before running.")

    # 生成训练/测试的 embedding 字典
    # train_embedding_dict: {case_name: embedding_vector}
    # test_embedding_dict: {case_name: embedding_vector}
    logkg.get_train_embedding()
    logkg.get_test_embedding()
    train_embedding = logkg.train_embedding_dict
    test_embedding = logkg.test_embedding_dict

    # 组装当前 fold 的 train/test 矩阵
    train_set = np.array([train_embedding[case_name_list[index]] for index in train_index])
    test_set = np.array([test_embedding[case_name_list[index]] for index in test_index])

    # 训练集聚类 + 每个簇选代表样本（返回的是“全数据索引”）
    cluster_centroids, cluster_result = get_logkg_result(
        train_set=train_set,
        train_index=train_index,
        verbose=verbose
    )

    # classify_index：给训练集每个样本分配一个“簇标签对应的真实类别”
    # 初始化为 -1 表示未分配/噪声
    classify_index = np.zeros(len(cluster_result), dtype=int) - 1

    # 遍历每个簇 i，把簇里所有样本的 pseudo-label 设置为该簇代表样本的真实标签
    for i in range(np.max(cluster_result) + 1):
        class_label = case_truth_label[cluster_centroids[i]]
        classify_index[np.where(cluster_result == i)[0]] = class_label

    # 只保留被赋值成功的样本（!= -1），构造监督训练数据
    classify_train_mask = classify_index != -1
    if not np.any(classify_train_mask):
        raise ValueError("No labeled samples after clustering.")

    classify_train_label = classify_index[classify_train_mask]
    classify_train_set = train_set[classify_train_mask]

    # 训练分类器（clf 和 clf_kwargs 是外部定义的）
    classifier = clf(**clf_kwargs)
    classifier.fit(classify_train_set, classify_train_label)

    # 测试集评估
    y_true = case_truth_label[test_index].astype(int)
    y_pred = classifier.predict(test_set).astype(int)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return y_true, y_pred, acc, macro_f1


# =========================
# 5) StratifiedKFold（分层K折）评估
# 关键点:
#   - 如果某个类别样本很少，n_splits 不能超过最小类别数
#   - 每折里重新构造 train_df/test_df（按 case_name -> log_df）
#   - 每折训练 LogKG 并评估，最后汇总均值/方差与混淆矩阵 confusion_matrix
# =========================
def run_stratified_kfold_eval(case_name_list, case_truth_label,
                              case_log_df, template_embedding,
                              n_splits=5, random_state=42,
                              verbose_fold=False):
    # 统计每类数量，避免分层K折时某类样本数 < n_splits
    class_counts = Counter(case_truth_label.tolist())
    min_class_count = min(class_counts.values())
    effective_splits = min(n_splits, min_class_count)
    if effective_splits < 2:
        raise ValueError(f"Not enough samples per class for StratifiedKFold: min_class_count={min_class_count}")

    skf = StratifiedKFold(n_splits=effective_splits, shuffle=True, random_state=random_state)

    acc_list = []
    f1_list = []
    y_true_all = []
    y_pred_all = []

    # 逐 fold 训练与测试
    for fold_idx, (train_index, test_index) in enumerate(
        skf.split(np.arange(len(case_name_list)), case_truth_label),
        start=1
    ):
        train_index = train_index.tolist()
        test_index = test_index.tolist()

        # 构造当前 fold 的训练/测试日志字典
        # train_df/test_df: {case_name: 对应的日志DataFrame}
        train_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in train_index}
        test_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in test_index}

        # 初始化 LogKG（兼容不同构造函数签名）
        try:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding, embedding_size=EMBEDDING_SIZE)
        except TypeError:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding)

        # 跑一折实验
        y_true, y_pred, acc, macro_f1 = LogKG_exp_run(
            case_name_list=case_name_list,
            case_truth_label=case_truth_label,
            train_index=train_index,
            test_index=test_index,
            logkg_config=model,
            verbose=verbose_fold,
        )

        acc_list.append(acc)
        f1_list.append(macro_f1)
        y_true_all.append(y_true)
        y_pred_all.append(y_pred)

        print(f"fold {fold_idx}/{effective_splits}: acc={acc:.4f}, macro_f1={macro_f1:.4f}, test_size={len(test_index)}")

    # 汇总所有 fold 的预测结果
    y_true_all = np.concatenate(y_true_all).astype(int)
    y_pred_all = np.concatenate(y_pred_all).astype(int)

    acc_mean = float(np.mean(acc_list))
    acc_std = float(np.std(acc_list))
    f1_mean = float(np.mean(f1_list))
    f1_std = float(np.std(f1_list))

    # 混淆矩阵：labels 用真实标签集合（保持顺序一致）
    labels = np.unique(case_truth_label)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

    print("=" * 30, "StratifiedKFold Summary", "=" * 30)
    print(f"accuracy: mean={acc_mean:.4f}, std={acc_std:.4f}")
    print(f"macro_f1: mean={f1_mean:.4f}, std={f1_std:.4f}")
    print("confusion_matrix labels:", labels.tolist())
    print(cm)

    # 返回结构化结果，便于外部保存/画图
    return {
        "acc_list": acc_list,
        "f1_list": f1_list,
        "acc_mean": acc_mean,
        "acc_std": acc_std,
        "f1_mean": f1_mean,
        "f1_std": f1_std,
        "labels": labels,
        "confusion_matrix": cm,
    }

In [ ]:
# Use LogKG class from code/model.py directly.
# This cell intentionally replaces the previous duplicated in-notebook class implementation.


In [ ]:
if DATASET_NAME == "CMCC":
    case_name_list = sorted([name.split(".")[0] for name in os.listdir(CASE_DIR)])
    config = json.load(open(CONFIG_PATH))
    case_truth_label = get_case_truth_label(case_name_list, config)
    case_log_df = {case_name: pd.read_csv(os.path.join(CASE_DIR, case_name + ".csv")) for case_name in case_name_list}

elif DATASET_NAME == "D1":
    d1_log_df = pd.read_csv(os.path.join(D1_DIR, D1_LOG_FILE))
    d1_label_df = pd.read_csv(os.path.join(D1_DIR, D1_LABEL_FILE))

    case_name_list, case_truth_label, case_log_df = build_d1_cases(
        log_df=d1_log_df,
        case_df=d1_label_df,
        label_col="label",
        history_hours=D1_HISTORY_HOURS,
        use_server_model=D1_USE_SERVER_MODEL,
        fallback_all_before=True,
    )

else:
    raise ValueError(f"Unsupported DATASET_NAME: {DATASET_NAME}")

if template_embedding is None:
    rng = np.random.default_rng(42)
    all_templates = sorted({eid for df in case_log_df.values() for eid in df["EventId"].values})
    template_embedding = {eid: rng.normal(0, 1, EMBEDDING_SIZE) for eid in all_templates}
    print("template_embedding initialized:", len(template_embedding))

print("DATASET_NAME:", DATASET_NAME)
print("case_num:", len(case_name_list))
print("class_dist:", Counter(case_truth_label.tolist()))


In [ ]:
print('-' * 30, "LogKG StratifiedKFold", '-' * 30)
eval_result = run_stratified_kfold_eval(
    case_name_list=case_name_list,
    case_truth_label=case_truth_label,
    case_log_df=case_log_df,
    template_embedding=template_embedding,
    n_splits=5,
    random_state=42,
    verbose_fold=False,
)
